# Build and Push Custom AutoGluon Image (PyTorch DLC Base)

This notebook builds custom training and inference Docker images for AutoGluon 1.5.0 on top of the PyTorch DLC base image.

Uses **finch** (not docker) for container builds.

## Configuration

In [ ]:
import subprocess
import boto3
from sagemaker import image_uris

# Configuration
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REPO_NAME = "autogluon-custom"
IMAGE_TAG = "ag150-pytorch-dlc"
PYTORCH_VERSION = "2.6"
PY_VERSION = "py312"

print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"Repository: {REPO_NAME}")
print(f"Image Tag: {IMAGE_TAG}")

## Resolve Base Images

In [ ]:
# Resolve PyTorch DLC base images
training_base_image = image_uris.retrieve(
    "pytorch",
    region=REGION,
    version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    image_scope="training",
    instance_type="ml.m5.2xlarge"
)

inference_base_image = image_uris.retrieve(
    "pytorch",
    region=REGION,
    version=PYTORCH_VERSION,
    py_version=PY_VERSION,
    image_scope="inference",
    instance_type="ml.m5.2xlarge"
)

print(f"Training base image: {training_base_image}")
print(f"Inference base image: {inference_base_image}")

## ECR Authentication

In [ ]:
# Login to DLC ECR account (for pulling base images)
dlc_account = training_base_image.split(".")[0]
print(f"Logging in to DLC ECR account: {dlc_account}")

result = subprocess.run(
    f"aws ecr get-login-password --region {REGION} | "
    f"finch login --username AWS --password-stdin {dlc_account}.dkr.ecr.{REGION}.amazonaws.com",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

# Login to user ECR account (for pushing custom images)
print(f"\nLogging in to user ECR account: {ACCOUNT_ID}")
result = subprocess.run(
    f"aws ecr get-login-password --region {REGION} | "
    f"finch login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

## Build Training Image

In [ ]:
training_image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-training"

print(f"Building training image: {training_image_uri}")
result = subprocess.run(
    f"cd ../docker && finch build "
    f"--build-arg BASE_IMAGE={training_base_image} "
    f"-t {training_image_uri} "
    f"-f Dockerfile.training .",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError("Training image build failed")

print(f"\nTraining image built successfully: {training_image_uri}")

## Build Inference Image

In [ ]:
inference_image_uri = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-inference"

print(f"Building inference image: {inference_image_uri}")
result = subprocess.run(
    f"cd ../docker && finch build "
    f"--build-arg BASE_IMAGE={inference_base_image} "
    f"-t {inference_image_uri} "
    f"-f Dockerfile.inference .",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError("Inference image build failed")

print(f"\nInference image built successfully: {inference_image_uri}")

## Smoke Test

In [ ]:
# Test training image
print("Testing training image...")
result = subprocess.run(
    f'finch run --rm {training_image_uri} python -c "import autogluon; print(autogluon.__version__)"',
    shell=True,
    capture_output=True,
    text=True
)
print(f"Training image AutoGluon version: {result.stdout.strip()}")
if result.returncode != 0:
    print(f"Error: {result.stderr}")

# Test inference image
print("\nTesting inference image...")
result = subprocess.run(
    f'finch run --rm {inference_image_uri} python -c "import autogluon; print(autogluon.__version__)"',
    shell=True,
    capture_output=True,
    text=True
)
print(f"Inference image AutoGluon version: {result.stdout.strip()}")
if result.returncode != 0:
    print(f"Error: {result.stderr}")

## Create ECR Repository

In [ ]:
ecr = boto3.client("ecr", region_name=REGION)

try:
    ecr.create_repository(repositoryName=REPO_NAME)
    print(f"Created ECR repository: {REPO_NAME}")
except ecr.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repository already exists: {REPO_NAME}")

## Push Images to ECR

In [ ]:
# Push training image
print(f"Pushing training image: {training_image_uri}")
result = subprocess.run(
    f"finch push {training_image_uri}",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError("Training image push failed")

# Push inference image
print(f"\nPushing inference image: {inference_image_uri}")
result = subprocess.run(
    f"finch push {inference_image_uri}",
    shell=True,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError("Inference image push failed")

## Summary

In [ ]:
print("\n" + "="*80)
print("BUILD AND PUSH COMPLETE")
print("="*80)
print(f"\nTraining Image URI:")
print(f"  {training_image_uri}")
print(f"\nInference Image URI:")
print(f"  {inference_image_uri}")
print(f"\nBase Images:")
print(f"  Training:  {training_base_image}")
print(f"  Inference: {inference_base_image}")
print("\nNext: Run ../1-training/launch_training.ipynb")